In [2]:
import pandas as pd
import numpy as np
import sqlite3
from scipy import stats

df = pd.read_csv("IBM-HR-Employee-Attrition.csv")

conn = sqlite3.connect(":memory:")
df.to_sql("employee_attrition", conn, index=False, if_exists="replace")

df.head()


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,4,80,1,6,3,3,2,2,2,2


In [4]:
#PHASE 1: Testing whether salary tier is statistically associated with attrition among long-commute employees.


In [6]:
df_far_commute = pd.read_sql_query("""
SELECT *
FROM employee_attrition
WHERE DistanceFromHome > 10
""", conn)

df_far_commute.head()


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,30,No,Travel_Rarely,1358,Research & Development,24,1,Life Sciences,1,11,...,2,80,1,1,2,3,1,0,0,0
1,38,No,Travel_Frequently,216,Research & Development,23,3,Life Sciences,1,12,...,2,80,0,10,2,3,9,7,1,8
2,36,No,Travel_Rarely,1299,Research & Development,27,3,Medical,1,13,...,2,80,2,17,3,2,7,7,7,7
3,35,No,Travel_Rarely,809,Research & Development,16,3,Medical,1,14,...,3,80,1,6,5,3,5,4,0,3
4,29,No,Travel_Rarely,153,Research & Development,15,2,Life Sciences,1,15,...,4,80,0,10,3,3,9,5,0,8


In [8]:
# Categorizing this group's monthly income into 4 pay tiers

pay_labels = ["Low Pay", "Mid-Low", "Mid-High", "High Pay"]

df_far_commute["Commute_Salary_Tier"] = pd.qcut(
    df_far_commute["MonthlyIncome"],
    q=4,
    labels=pay_labels
)


In [10]:
commute_paradox_matrix = pd.crosstab(
    df_far_commute["Commute_Salary_Tier"],
    df_far_commute["Attrition"],
    normalize="index"
) * 100

print("=== COMMUTE COMPENSATION ANALYSIS ===")
print(commute_paradox_matrix.round(1).astype(str) + "%")


=== COMMUTE COMPENSATION ANALYSIS ===
Attrition               No    Yes
Commute_Salary_Tier              
Low Pay              65.8%  34.2%
Mid-Low              83.8%  16.2%
Mid-High             81.1%  18.9%
High Pay             85.6%  14.4%


In [12]:
commute_contingency = pd.crosstab(
    df_far_commute['Commute_Salary_Tier'],
    df_far_commute['Attrition']
)

chi2_stat, p_value, dof, expected = stats.chi2_contingency(commute_contingency)

print("=== COMPENSATION PHASE 1: COMMUTE COMPENSATION VALIDATION ===")
print(f"Chi-Square Statistic : {chi2_stat:.4f}")
print(f"Degrees of Freedom   : {dof}")
print(f"P-Value              : {p_value:.6f}")

print("\n=== STRATEGIC VERIFICATION DECISION ===")
if p_value < 0.05:
    print("REJECT NULL HYPOTHESIS: Salary tier is statistically associated with attrition among long-commute employees.")
    print("Compensation level appears related to attrition outcomes for this commute-risk cohort.")
else:
    print("FAIL TO REJECT NULL HYPOTHESIS: Attrition rates do not differ significantly across salary tiers for long-commute employees.")
    print("This dataset does not provide strong evidence that salary tier is associated with attrition in this cohort.")


=== COMPENSATION PHASE 1: COMMUTE COMPENSATION VALIDATION ===
Chi-Square Statistic : 16.4716
Degrees of Freedom   : 3
P-Value              : 0.000907

=== STRATEGIC VERIFICATION DECISION ===
REJECT NULL HYPOTHESIS: Salary tier is statistically associated with attrition among long-commute employees.
Compensation level appears related to attrition outcomes for this commute-risk cohort.


In [14]:
#PHASE 2: Testing whether stock option level is statistically associated with attrition among overtime employees within each role.


In [16]:
tesla_role_mapping = {
    "Healthcare Representative": "EHS Specialist",
    "Human Resources": "People Engineering Partner",
    "Laboratory Technician": "Test Technician (Battery/Drive Unit)",
    "Manager": "Production Supervisor / Eng Manager",
    "Manufacturing Director": "Gigafactory Production Director",
    "Research Director": "Principal Staff Scientist",
    "Research Scientist": "Battery R&D / AI Engineer",
    "Sales Executive": "Tesla Advisor / Account Manager",
    "Sales Representative": "Product Specialist (Retail/Energy)"
}

df["Tesla_JobRole"] = df["JobRole"].replace(tesla_role_mapping)

conn = sqlite3.connect(":memory:")
df.to_sql("employee_attrition", conn, index=False, if_exists="replace")


1470

In [18]:
df_overtime = pd.read_sql_query("""
SELECT *
FROM employee_attrition
WHERE OverTime = 'Yes'
""", conn)

df_overtime.head()


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager,Tesla_JobRole
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,80,0,8,0,1,6,4,0,5,Tesla Advisor / Account Manager
1,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,80,0,7,3,3,0,0,0,0,Test Technician (Battery/Drive Unit)
2,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,80,0,8,3,3,8,7,3,0,Battery R&D / AI Engineer
3,59,No,Travel_Rarely,1324,Research & Development,3,3,Medical,1,10,...,80,3,12,3,2,1,0,0,0,Test Technician (Battery/Drive Unit)
4,29,No,Travel_Rarely,153,Research & Development,15,2,Life Sciences,1,15,...,80,0,10,3,3,9,5,0,8,Test Technician (Battery/Drive Unit)


In [20]:
role_equity_matrix = pd.crosstab(
    [df_overtime["Tesla_JobRole"], df_overtime["StockOptionLevel"]],
    df_overtime["Attrition"],
    normalize="index"
) * 100

print("=== ROLE-SPECIFIC EQUITY ANALYSIS ===")
print(role_equity_matrix.round(1).astype(str) + "%")


=== ROLE-SPECIFIC EQUITY ANALYSIS ===
Attrition                                                  No     Yes
Tesla_JobRole                        StockOptionLevel                
Battery R&D / AI Engineer            0                  54.3%   45.7%
                                     1                  77.1%   22.9%
                                     2                 100.0%    0.0%
                                     3                  50.0%   50.0%
EHS Specialist                       0                  85.7%   14.3%
                                     1                 100.0%    0.0%
                                     2                 100.0%    0.0%
Gigafactory Production Director      0                  88.9%   11.1%
                                     1                  93.3%    6.7%
                                     2                  80.0%   20.0%
                                     3                 100.0%    0.0%
People Engineering Partner           0              

In [21]:
for role in df_overtime["Tesla_JobRole"].dropna().unique():
    role_slice = df_overtime[df_overtime["Tesla_JobRole"] == role]

    contingency = pd.crosstab(role_slice["StockOptionLevel"], role_slice["Attrition"])

    print(f"Job Role: {role}")

    if contingency.shape[0] > 1 and contingency.shape[1] > 1:
        _, p_val, _, _ = stats.chi2_contingency(contingency)

        if p_val < 0.05:
            print("  VERDICT: Stock option level is statistically associated with attrition in this overtime role group.")
        else:
            print("  VERDICT: This sample does not provide strong evidence that stock option level is associated with attrition in this overtime role group.")
    else:
        print("  VERDICT: Not enough variation or sample size to test this role.")

    print("-" * 50)


Job Role: Tesla Advisor / Account Manager
  VERDICT: Stock option level is statistically associated with attrition in this overtime role group.
--------------------------------------------------
Job Role: Test Technician (Battery/Drive Unit)
  VERDICT: Stock option level is statistically associated with attrition in this overtime role group.
--------------------------------------------------
Job Role: Battery R&D / AI Engineer
  VERDICT: Stock option level is statistically associated with attrition in this overtime role group.
--------------------------------------------------
Job Role: EHS Specialist
  VERDICT: This sample does not provide strong evidence that stock option level is associated with attrition in this overtime role group.
--------------------------------------------------
Job Role: Product Specialist (Retail/Energy)
  VERDICT: This sample does not provide strong evidence that stock option level is associated with attrition in this overtime role group.
-------------------